# HMCN-F Ablation — Effect of β

$$P_F = \beta \cdot [P_{L1} \mid P_{L2}] + (1 - \beta) \cdot P_G$$

- **β = 1.0** → inference uses only local heads
- **β = 0.0** → inference uses only the global head
- **β = 0.5** → equal mix (current default)

## Fixed parameters

| Parameter | Value |
|---|---|
| global_dim | 128 |
| local_dim | 64 |
| dropout | 0.47 |
| lr | 1e-4 |
| weight_decay | 1e-4 |
| lambda_viol | 0.1 |

## Grid
β ∈ {0.1, 0.3, 0.5, 0.7, 0.9, 1.0}

All metrics computed via `hmcn_eval.py` and appended to `hmcn_ablation_results.csv`.

## 1. Install

In [ ]:
!pip install iterative-stratification -q

## 2. Imports

`hmcn_eval.py` must be in the same directory as this notebook (or on the Colab path).

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
import warnings
warnings.filterwarnings('ignore')

# Shared evaluation utility
from hmcn_eval import (
    find_optimal_thresholds,
    compute_all_metrics,
    compute_test_loss,
    save_experiment,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 3. Fixed hyperparameters and ablation grid

In [ ]:
GLOBAL_DIM   = 128
LOCAL_DIM    = 64
DROPOUT      = 0.47
LR           = 1e-4
WEIGHT_DECAY = 1e-4
LAMBDA_VIOL  = 0.1
BATCH_SIZE   = 32
EPOCHS       = 300
PATIENCE     = 40
T_0_EPOCHS   = 50
T_MULT       = 2
ETA_MIN      = 1e-5
SEED         = 42

BETA_VALUES  = [0.1, 0.3, 0.5, 0.7, 0.9, 1.0]

RESULTS_CSV  = 'hmcn_ablation_results.csv'
EXPERIMENT   = 'beta_ablation'

## 4. Hierarchy — Vassilis's lookup table

In [ ]:
META_CATEGORIES = {
    'floral':        ['floral','rose','jasmin','lily','muguet','violet','hyacinth',
                      'geranium','lavender','orangeflower','chamomile','hawthorn'],
    'fruity':        ['fruity','apple','apricot','banana','berry','cherry','grape',
                      'grapefruit','lemon','melon','orange','peach','pear','pineapple',
                      'plum','raspberry','strawberry','tropical','black currant','fruit skin'],
    'sweet':         ['sweet','vanilla','caramellic','honey','chocolate','cocoa',
                      'coconut','creamy','buttery','milky','dairy'],
    'woody':         ['woody','cedar','sandalwood','pine','vetiver','terpenic',
                      'balsamic','cortex'],
    'green':         ['green','grassy','herbal','leafy','hay','tea','fresh',
                      'cucumber','vegetable','weedy'],
    'spicy':         ['spicy','cinnamon','clove','warm','pungent','sharp',
                      'cooling','mint','camphoreous'],
    'animal_musk':   ['animal','musk','leathery','fishy','sweaty','meaty',
                      'beefy','musty'],
    'earthy':        ['earthy','mushroom','nutty','hazelnut','roasted','coffee',
                      'tobacco','smoky','popcorn'],
    'citrus':        ['citrus','bergamot','ozone','clean','soapy'],
    'chemical':      ['solvent','ethereal','metallic','medicinal','phenolic',
                      'sulfurous','gassy','burnt','oily'],
    'gourmand':      ['almond','malty','rummy','brandy','cognac','winey','cooked',
                      'potato','savory','celery','tomato','radish','onion','garlic',
                      'cabbage','cheesy'],
    'powdery_amber': ['amber','powdery','anisic','coumarinic','orris','waxy',
                      'aldehydic','ketonic','lactonic'],
}

## 5. Data loading, splitting, scaling

In [ ]:
def load_data(csv_path='hmcn_dataset.csv'):
    df = pd.read_csv(csv_path)
    fine_cols = [c for c in df.columns if c.startswith('fine_')]
    meta_cols = [c for c in df.columns if c.startswith('meta_')]
    feat_cols = [c for c in df.columns
                 if c not in fine_cols + meta_cols + ['SMILES']]
    X          = df[feat_cols].values.astype(np.float32)
    Y1         = df[fine_cols].values.astype(np.float32)
    Y2         = df[meta_cols].values.astype(np.float32)
    fine_names = [c.replace('fine_', '') for c in fine_cols]
    meta_names = [c.replace('meta_', '') for c in meta_cols]
    return X, Y1, Y2, fine_names, meta_names


def split_and_scale(X, Y1, Y2, seed=SEED):
    msss1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    trainval_idx, test_idx = next(msss1.split(X, Y1))

    msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.1/0.8, random_state=seed)
    train_idx, val_idx = next(msss2.split(X[trainval_idx], Y1[trainval_idx]))

    X_train  = X[trainval_idx][train_idx]
    X_val    = X[trainval_idx][val_idx]
    X_test   = X[test_idx]
    Y1_train = Y1[trainval_idx][train_idx]
    Y1_val   = Y1[trainval_idx][val_idx]
    Y1_test  = Y1[test_idx]
    Y2_train = Y2[trainval_idx][train_idx]
    Y2_val   = Y2[trainval_idx][val_idx]
    Y2_test  = Y2[test_idx]

    scaler  = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val   = scaler.transform(X_val)
    X_test  = scaler.transform(X_test)

    return (X_train, Y1_train, Y2_train,
            X_val,   Y1_val,   Y2_val,
            X_test,  Y1_test,  Y2_test)


def build_violation_pairs(fine_names, meta_names):
    fine_idx = {name: i for i, name in enumerate(fine_names)}
    meta_idx = {name: i for i, name in enumerate(meta_names)}
    pairs = []
    for meta, members in META_CATEGORIES.items():
        for member in members:
            if member in fine_idx and meta in meta_idx:
                pairs.append((fine_idx[member], meta_idx[meta]))
    return pairs


# Load once — reused across all beta runs
X, Y1, Y2, fine_names, meta_names = load_data('hmcn_dataset.csv')
N_FINE          = Y1.shape[1]   # 138
N_META          = Y2.shape[1]   # 12
violation_pairs = build_violation_pairs(fine_names, meta_names)

(
    X_train, Y1_train, Y2_train,
    X_val,   Y1_val,   Y2_val,
    X_test,  Y1_test,  Y2_test
) = split_and_scale(X, Y1, Y2)

print(f'Train : {len(X_train):,} molecules')
print(f'Val   : {len(X_val):,} molecules')
print(f'Test  : {len(X_test):,} molecules')
print(f'Features        : {X_train.shape[1]}')
print(f'Fine labels     : {N_FINE}')
print(f'Meta labels     : {N_META}')
print(f'Hierarchy pairs : {len(violation_pairs)}')

## 6. DataLoaders

In [ ]:
def make_loader(X, Y1, Y2, batch_size, shuffle):
    dataset = TensorDataset(
        torch.tensor(X,  dtype=torch.float32),
        torch.tensor(Y1, dtype=torch.float32),
        torch.tensor(Y2, dtype=torch.float32)
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


train_loader = make_loader(X_train, Y1_train, Y2_train, BATCH_SIZE, shuffle=True)
val_loader   = make_loader(X_val,   Y1_val,   Y2_val,   128,        shuffle=False)
test_loader  = make_loader(X_test,  Y1_test,  Y2_test,  128,        shuffle=False)

## 7. Model definition

In [ ]:
class LocalBlock(nn.Module):
    def __init__(self, input_dim, global_dim, local_dim, n_labels, dropout):
        super().__init__()
        self.global_fc = nn.Sequential(
            nn.Linear(global_dim + input_dim, global_dim),
            nn.BatchNorm1d(global_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.transition = nn.Sequential(
            nn.Linear(global_dim, local_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.output = nn.Linear(local_dim, n_labels)

    def forward(self, x, A_G):
        A_G_next = self.global_fc(torch.cat([A_G, x], dim=1))
        A_L      = self.transition(A_G_next)
        P_L      = torch.sigmoid(self.output(A_L))
        return A_G_next, P_L


class HMCNF(nn.Module):
    def __init__(self, input_dim, n_fine, n_meta,
                 global_dim, local_dim, dropout, beta):
        super().__init__()
        self.beta    = beta
        self.n_fine  = n_fine
        self.n_total = n_fine + n_meta

        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, global_dim),
            nn.BatchNorm1d(global_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.level1        = LocalBlock(input_dim, global_dim, local_dim, n_fine, dropout)
        self.level2        = LocalBlock(input_dim, global_dim, local_dim, n_meta, dropout)
        self.global_output = nn.Linear(global_dim, self.n_total)

    def forward(self, x):
        A_G       = self.input_proj(x)
        A_G, P_L1 = self.level1(x, A_G)
        A_G, P_L2 = self.level2(x, A_G)
        P_G       = torch.sigmoid(self.global_output(A_G))
        P_F       = self.beta * torch.cat([P_L1, P_L2], dim=1) + (1 - self.beta) * P_G
        return P_F, P_L1, P_L2, P_G

## 8. Loss function

In [ ]:
def binary_cross_entropy(P, Y, eps=1e-7):
    P = torch.clamp(P, eps, 1 - eps)
    return -torch.mean(Y * torch.log(P) + (1 - Y) * torch.log(1 - P))


def hierarchical_violation_penalty(P_L1, P_L2, pairs):
    total = torch.tensor(0.0, device=P_L1.device)
    for fi, mi in pairs:
        v     = torch.clamp(P_L1[:, fi] - P_L2[:, mi], min=0.0)
        total = total + torch.mean(v ** 2)
    return total / max(len(pairs), 1)


def hmcn_loss(P_F, P_L1, P_L2, P_G, Y1, Y2, pairs, lambda_viol):
    Y_global = torch.cat([Y1, Y2], dim=1)
    return (
        binary_cross_entropy(P_L1, Y1)
        + binary_cross_entropy(P_L2, Y2)
        + binary_cross_entropy(P_G, Y_global)
        + lambda_viol * hierarchical_violation_penalty(P_L1, P_L2, pairs)
    )

## 9. Training and prediction utilities

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler, pairs, lambda_viol, device):
    model.train()
    total_loss = 0.0
    for X_batch, Y1_batch, Y2_batch in loader:
        X_batch  = X_batch.to(device)
        Y1_batch = Y1_batch.to(device)
        Y2_batch = Y2_batch.to(device)
        optimizer.zero_grad()
        P_F, P_L1, P_L2, P_G = model(X_batch)
        loss = hmcn_loss(P_F, P_L1, P_L2, P_G, Y1_batch, Y2_batch, pairs, lambda_viol)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def collect_predictions(model, loader, device, n_fine):
    """
    Returns P_F split into fine/meta portions.
    P_F[:, :n_fine] → fine predictions
    P_F[:, n_fine:] → meta predictions
    """
    model.eval()
    fine_probs, fine_true = [], []
    meta_probs, meta_true = [], []
    for X_batch, Y1_batch, Y2_batch in loader:
        P_F, _, _, _ = model(X_batch.to(device))
        fine_probs.append(P_F[:, :n_fine].cpu().numpy())
        meta_probs.append(P_F[:, n_fine:].cpu().numpy())
        fine_true.append(Y1_batch.numpy())
        meta_true.append(Y2_batch.numpy())
    return (
        np.vstack(fine_probs), np.vstack(fine_true),
        np.vstack(meta_probs), np.vstack(meta_true)
    )

## 10. Beta ablation loop

In [ ]:
ablation_results = []

print(f"{'β':>5}  {'best_ep':>8}  {'val_AUC':>9}  {'pr_auc_12':>10}  "
      f"{'roc_auc_12':>11}  {'f1_macro_12':>12}  {'jaccard_12':>11}  {'hamming_12':>11}")
print('─' * 90)

for beta in BETA_VALUES:

    # ── Reproducibility ───────────────────────────────────────────────────────
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    np.random.seed(SEED)

    # ── Fresh model ───────────────────────────────────────────────────────────
    model = HMCNF(
        input_dim  = X_train.shape[1],
        n_fine     = N_FINE,
        n_meta     = N_META,
        global_dim = GLOBAL_DIM,
        local_dim  = LOCAL_DIM,
        dropout    = DROPOUT,
        beta       = beta
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer,
        T_0     = T_0_EPOCHS * len(train_loader),
        T_mult  = T_MULT,
        eta_min = ETA_MIN
    )

    # ── Training loop ─────────────────────────────────────────────────────────
    best_val_auc      = 0.0
    best_model_state  = None
    best_train_loss   = 0.0
    patience_counter  = 0
    best_epoch        = 1

    for epoch in range(1, EPOCHS + 1):
        train_loss = train_one_epoch(
            model, train_loader, optimizer, scheduler,
            violation_pairs, LAMBDA_VIOL, device
        )
        fp_v, ft_v, mp_v, mt_v = collect_predictions(model, val_loader, device, N_FINE)
        val_meta_auc = float(np.mean([
            roc_auc_score(mt_v[:, i], mp_v[:, i])
            for i in range(N_META) if mt_v[:, i].sum() > 0
        ]))

        if val_meta_auc > best_val_auc:
            best_val_auc     = val_meta_auc
            best_train_loss  = train_loss
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
            best_epoch       = epoch
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                break

    # ── Load best checkpoint ──────────────────────────────────────────────────
    model.load_state_dict(best_model_state)
    model.to(device)

    # ── Calibrate thresholds on val ───────────────────────────────────────────
    fp_v, ft_v, mp_v, mt_v = collect_predictions(model, val_loader, device, N_FINE)
    fine_thresholds = find_optimal_thresholds(fp_v, ft_v)
    meta_thresholds = find_optimal_thresholds(mp_v, mt_v)

    # ── Test predictions ──────────────────────────────────────────────────────
    fp_t, ft_t, mp_t, mt_t = collect_predictions(model, test_loader, device, N_FINE)

    # ── Test loss ─────────────────────────────────────────────────────────────
    test_loss = compute_test_loss(model, test_loader, violation_pairs, LAMBDA_VIOL, device)

    # ── All metrics via hmcn_eval ─────────────────────────────────────────────
    metrics = compute_all_metrics(
        fine_probs      = fp_t,
        fine_true       = ft_t,
        meta_probs      = mp_t,
        meta_true       = mt_t,
        meta_thresholds = meta_thresholds,
        fine_thresholds = fine_thresholds,
        violation_pairs = violation_pairs,
        meta_names      = meta_names,
        fine_names      = fine_names,
        Y2_train        = Y2_train,
        test_loss       = test_loss,
    )

    # ── Print summary row ─────────────────────────────────────────────────────
    print(
        f"{beta:>5.1f}  {best_epoch:>8d}  {best_val_auc:>9.4f}  "
        f"{metrics['pr_auc_12']:>10.4f}  {metrics['roc_auc_12']:>11.4f}  "
        f"{metrics['f1_macro_12']:>12.4f}  {metrics['jaccard_12']:>11.4f}  "
        f"{metrics['hamming_loss_12']:>11.4f}"
    )

    # ── Build config and save ─────────────────────────────────────────────────
    config = dict(
        experiment       = EXPERIMENT,
        param_name       = 'beta',
        param_value      = beta,
        global_dim       = GLOBAL_DIM,
        local_dim        = LOCAL_DIM,
        dropout          = DROPOUT,
        lr               = LR,
        weight_decay     = WEIGHT_DECAY,
        lambda_viol      = LAMBDA_VIOL,
        beta             = beta,
        batch_size       = BATCH_SIZE,
        seed             = SEED,
        best_epoch       = best_epoch,
        train_loss_at_best = round(best_train_loss, 4),
        val_meta_roc_auc = round(best_val_auc, 4),
    )
    save_experiment(config, metrics, csv_path=RESULTS_CSV)
    ablation_results.append({**config, **metrics})


print()
print('Done.')

## 11. Summary table

Sorted by `pr_auc_12` (primary reporting metric).

In [ ]:
results_df = pd.DataFrame(ablation_results).sort_values('pr_auc_12', ascending=False)

display_cols = [
    'param_value', 'best_epoch', 'val_meta_roc_auc',
    # meta (12)
    'roc_auc_12', 'pr_auc_12', 'f1_macro_12', 'instance_f1_12',
    'balanced_accuracy_12', 'sensitivity_macro_12', 'specificity_macro_12',
    'jaccard_12', 'hamming_loss_12',
    'hier_violation_rate_12', 'label_cooc_consistency_12',
    # fine (138)
    'pr_auc_138', 'f1_macro_138', 'jaccard_138', 'hamming_loss_138',
    'hier_violation_rate_138',
    # loss
    'test_loss',
]

print('BETA ABLATION — sorted by pr_auc_12')
print(results_df[display_cols].rename(columns={'param_value': 'beta'}).to_string(index=False))

best_beta = results_df.iloc[0]['param_value']
print(f'\nBest β = {best_beta}')
print(f'Results saved to: {RESULTS_CSV}')

## 12. Interpretation guide

| Result pattern | Interpretation |
|---|---|
| Best β ≈ 0.0 | Global head dominates — local outputs add noise on 2-level hierarchy |
| Best β ≈ 1.0 | Local heads are strong, global head is the bottleneck |
| Flat curve | β barely matters — bottleneck is elsewhere (capacity, data size) |
| Peak at 0.5 | Both flows contribute — original default was well-calibrated |